<a href="https://colab.research.google.com/github/vivek28n/Medical-RAG-Hallucination-Detection/blob/main/notebooks/Notebook_05_Hallucination_Detection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ============================================
# NOTEBOOK 05 - HALLUCINATION DETECTION
# ============================================

print("Notebook 5 - Hallucination Detection")
print("Starting setup...")

Notebook 5 - Hallucination Detection
Starting setup...


In [2]:
!pip install -q sentence-transformers transformers torch

In [3]:
import numpy as np
import torch

from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForSequenceClassification

print("✅ Libraries imported successfully")

✅ Libraries imported successfully


In [4]:
# ============================================
# LOAD SEMANTIC SIMILARITY MODEL
# ============================================

embedding_model = SentenceTransformer(
    "all-MiniLM-L6-v2"
)

print("✅ Semantic similarity model loaded")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

✅ Semantic similarity model loaded


In [5]:
# ============================================
# LOAD NLI MODEL
# ============================================

NLI_MODEL = "cross-encoder/nli-deberta-v3-base"

nli_tokenizer = AutoTokenizer.from_pretrained(NLI_MODEL)

nli_model = AutoModelForSequenceClassification.from_pretrained(
    NLI_MODEL
)

nli_model.eval()

print("✅ NLI model loaded")

config.json:   0%|          | 0.00/1.05k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.35k [00:00<?, ?B/s]

spm.model: reconstructing file:   0%|          |  0.00B / 2.46MB            

spm.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/8.66M [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/301 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  738MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

✅ NLI model loaded


In [6]:
# ============================================
# SEMANTIC SIMILARITY
# ============================================

def semantic_similarity(answer, retrieved_documents):
    """
    Compare the generated answer with retrieved evidence.

    Returns:
        max_similarity: highest similarity between answer
                        and any retrieved document.
        similarities: similarity score for each document.
    """

    # Generate embedding for the answer
    answer_embedding = embedding_model.encode(
        answer,
        normalize_embeddings=True
    )

    # Generate embeddings for retrieved evidence
    document_texts = [
        doc["text"] for doc in retrieved_documents
    ]

    document_embeddings = embedding_model.encode(
        document_texts,
        normalize_embeddings=True
    )

    # Cosine similarity
    similarities = np.dot(
        document_embeddings,
        answer_embedding
    )

    # Highest matching evidence
    max_similarity = float(np.max(similarities))

    return max_similarity, similarities


print("✅ Semantic similarity function created")

✅ Semantic similarity function created


In [7]:
# ============================================
# TEST SEMANTIC SIMILARITY
# ============================================

test_documents = [
    {
        "text": "Obesity and physical inactivity are risk factors for type 2 diabetes."
    },
    {
        "text": "Family history of diabetes can increase the risk of developing diabetes."
    }
]

supported_answer = (
    "Obesity and lack of physical activity can increase "
    "the risk of type 2 diabetes."
)

score, all_scores = semantic_similarity(
    supported_answer,
    test_documents
)

print("Answer:")
print(supported_answer)

print("\nSimilarity scores:")
print(all_scores)

print("\nMaximum similarity:")
print(round(score, 4))

Answer:
Obesity and lack of physical activity can increase the risk of type 2 diabetes.

Similarity scores:
[0.9007923 0.5450779]

Maximum similarity:
0.9008


In [8]:
# ============================================
# NLI ENTAILMENT CHECK
# ============================================

def nli_score(premise, hypothesis):
    """
    Check whether the premise supports the hypothesis.

    Returns:
        entailment_score
        neutral_score
        contradiction_score
    """

    inputs = nli_tokenizer(
        premise,
        hypothesis,
        return_tensors="pt",
        truncation=True,
        max_length=512
    )

    with torch.no_grad():
        outputs = nli_model(**inputs)

    probabilities = torch.softmax(
        outputs.logits,
        dim=-1
    )[0]

    # NLI-DeBERTa labels:
    # 0 = contradiction
    # 1 = entailment
    # 2 = neutral

    contradiction_score = float(probabilities[0])
    entailment_score = float(probabilities[1])
    neutral_score = float(probabilities[2])

    return (
        entailment_score,
        neutral_score,
        contradiction_score
    )


print("✅ NLI function created")

✅ NLI function created


In [9]:
# ============================================
# TEST NLI
# ============================================

premise = (
    "Obesity and physical inactivity are risk factors "
    "for type 2 diabetes."
)

hypothesis = (
    "Obesity and lack of physical activity can increase "
    "the risk of type 2 diabetes."
)

entailment, neutral, contradiction = nli_score(
    premise,
    hypothesis
)

print("Premise:")
print(premise)

print("\nHypothesis:")
print(hypothesis)

print("\nNLI Scores:")
print("Entailment   :", round(entailment, 4))
print("Neutral      :", round(neutral, 4))
print("Contradiction:", round(contradiction, 4))

Premise:
Obesity and physical inactivity are risk factors for type 2 diabetes.

Hypothesis:
Obesity and lack of physical activity can increase the risk of type 2 diabetes.

NLI Scores:
Entailment   : 0.9957
Neutral      : 0.0042
Contradiction: 0.0001


In [10]:
# ============================================
# NLI CHECK AGAINST RETRIEVED EVIDENCE
# ============================================

def check_nli_against_documents(answer, retrieved_documents):
    """
    Check generated answer against every retrieved document.

    Returns:
        A list containing NLI scores for each document.
    """

    results = []

    for rank, doc in enumerate(retrieved_documents, start=1):

        entailment, neutral, contradiction = nli_score(
            doc["text"],
            answer
        )

        results.append({
            "rank": rank,
            "page": doc.get("page"),
            "entailment": entailment,
            "neutral": neutral,
            "contradiction": contradiction
        })

    return results


print("✅ Evidence-level NLI function created")

✅ Evidence-level NLI function created


In [11]:
# ============================================
# TEST EVIDENCE-LEVEL NLI
# ============================================

nli_results = check_nli_against_documents(
    supported_answer,
    test_documents
)

for result in nli_results:
    print(
        f"Evidence {result['rank']} | "
        f"Entailment: {result['entailment']:.4f} | "
        f"Neutral: {result['neutral']:.4f} | "
        f"Contradiction: {result['contradiction']:.4f}"
    )

Evidence 1 | Entailment: 0.9957 | Neutral: 0.0042 | Contradiction: 0.0001
Evidence 2 | Entailment: 0.0001 | Neutral: 0.9993 | Contradiction: 0.0006


In [12]:
def get_max_entailment(nli_results):
    """
    Get the strongest entailment score
    among all retrieved evidence.
    """
    return max(result["entailment"] for result in nli_results)


max_entailment = get_max_entailment(nli_results)

print("Maximum Entailment Score:", round(max_entailment, 4))

Maximum Entailment Score: 0.9957


In [13]:
def calculate_support_score(similarity_score, entailment_score):
    """
    Combine semantic similarity and NLI entailment
    into one evidence support score.
    """
    support_score = (
        0.5 * similarity_score +
        0.5 * entailment_score
    )

    return support_score


support_score = calculate_support_score(
    score,
    max_entailment
)

print("Semantic Similarity :", round(score, 4))
print("NLI Entailment      :", round(max_entailment, 4))
print("Evidence Support    :", round(support_score, 4))

Semantic Similarity : 0.9008
NLI Entailment      : 0.9957
Evidence Support    : 0.9482


In [14]:
SUPPORT_THRESHOLD = 0.60
CONTRADICTION_THRESHOLD = 0.50


def detect_hallucination(
    support_score,
    nli_results,
    support_threshold=SUPPORT_THRESHOLD,
    contradiction_threshold=CONTRADICTION_THRESHOLD
):
    """
    Decide whether the answer is supported by the evidence.
    """

    max_contradiction = max(
        result["contradiction"]
        for result in nli_results
    )

    if max_contradiction >= contradiction_threshold:
        return "CONTRADICTED"

    elif support_score >= support_threshold:
        return "SUPPORTED"

    else:
        return "POTENTIAL HALLUCINATION"


decision = detect_hallucination(
    support_score,
    nli_results
)

print("Hallucination Detection Result:", decision)

Hallucination Detection Result: SUPPORTED


In [15]:
hallucinated_answer = (
    "Type 2 diabetes is caused only by eating sugar, "
    "and obesity and physical inactivity do not affect "
    "the risk of developing it."
)

hallucinated_similarity, _ = semantic_similarity(
    hallucinated_answer,
    test_documents
)

hallucinated_nli_results = check_nli_against_documents(
    hallucinated_answer,
    test_documents
)

hallucinated_entailment = get_max_entailment(
    hallucinated_nli_results
)

hallucinated_support = calculate_support_score(
    hallucinated_similarity,
    hallucinated_entailment
)

hallucinated_decision = detect_hallucination(
    hallucinated_support,
    hallucinated_nli_results
)

print("Hallucinated Answer:")
print(hallucinated_answer)

print("\nSemantic Similarity:",
      round(hallucinated_similarity, 4))

print("NLI Entailment:",
      round(hallucinated_entailment, 4))

print("Evidence Support:",
      round(hallucinated_support, 4))

print("Detection Result:",
      hallucinated_decision)

Hallucinated Answer:
Type 2 diabetes is caused only by eating sugar, and obesity and physical inactivity do not affect the risk of developing it.

Semantic Similarity: 0.8628
NLI Entailment: 0.0
Evidence Support: 0.4314
Detection Result: CONTRADICTED


In [16]:
def analyze_answer(answer, retrieved_documents):
    """
    Analyze an answer for evidence support
    using semantic similarity and NLI.
    """

    # 1. Semantic similarity
    similarity_score, similarities = semantic_similarity(
        answer,
        retrieved_documents
    )

    # 2. NLI against all evidence
    nli_results = check_nli_against_documents(
        answer,
        retrieved_documents
    )

    # 3. Strongest entailment
    entailment_score = get_max_entailment(
        nli_results
    )

    # 4. Combined support score
    support_score = calculate_support_score(
        similarity_score,
        entailment_score
    )

    # 5. Final decision
    decision = detect_hallucination(
        support_score,
        nli_results
    )

    return {
        "semantic_similarity": similarity_score,
        "nli_entailment": entailment_score,
        "support_score": support_score,
        "decision": decision,
        "nli_results": nli_results
    }


print("✅ Complete hallucination analysis function created")

✅ Complete hallucination analysis function created


In [17]:
# Test 1: Supported answer
supported_result = analyze_answer(
    supported_answer,
    test_documents
)

# Test 2: Hallucinated answer
hallucinated_result = analyze_answer(
    hallucinated_answer,
    test_documents
)

print("========== SUPPORTED ANSWER ==========")
print("Semantic Similarity:",
      round(supported_result["semantic_similarity"], 4))
print("NLI Entailment:",
      round(supported_result["nli_entailment"], 4))
print("Evidence Support:",
      round(supported_result["support_score"], 4))
print("Decision:",
      supported_result["decision"])


print("\n========== HALLUCINATED ANSWER ==========")
print("Semantic Similarity:",
      round(hallucinated_result["semantic_similarity"], 4))
print("NLI Entailment:",
      round(hallucinated_result["nli_entailment"], 4))
print("Evidence Support:",
      round(hallucinated_result["support_score"], 4))
print("Decision:",
      hallucinated_result["decision"])

========== SUPPORTED ANSWER ==========
Semantic Similarity: 0.9008
NLI Entailment: 0.9957
Evidence Support: 0.9482
Decision: SUPPORTED

========== HALLUCINATED ANSWER ==========
Semantic Similarity: 0.8628
NLI Entailment: 0.0
Evidence Support: 0.4314
Decision: CONTRADICTED


In [18]:
print("chunks:", "chunks" in globals())
print("embedding_model:", "embedding_model" in globals())
print("index:", "index" in globals())
print("client:", "client" in globals())

chunks: False
embedding_model: True
index: False
client: False


In [20]:
!git clone https://github.com/vivek28n/Medical-RAG-Hallucination-Detection.git

print("✅ Repository cloned successfully")

Cloning into 'Medical-RAG-Hallucination-Detection'...
remote: Enumerating objects: 87, done.
remote: Counting objects: 100% (87/87), done.
remote: Compressing objects: 100% (75/75), done.
remote: Total 87 (delta 44), reused 36 (delta 10), pack-reused 0 (from 0)
Receiving objects: 100% (87/87), 1.39 MiB | 15.02 MiB/s, done.
Resolving deltas: 100% (44/44), done.
✅ Repository cloned successfully


In [21]:
import os

PDF_PATH = "/content/Medical-RAG-Hallucination-Detection/dataset/raw/niddk_guiding_principles_diabetes.pdf"

print("PDF exists:", os.path.exists(PDF_PATH))

PDF exists: True


In [24]:
!pip install -q PyPDF2
print("✅ PyPDF2 installed successfully")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.6/232.6 kB 5.2 MB/s eta 0:00:00
✅ PyPDF2 installed successfully


In [25]:
import PyPDF2

pdf_reader = PyPDF2.PdfReader(PDF_PATH)

pages = []

for page in pdf_reader.pages:
    text = page.extract_text()
    pages.append(text if text else "")

print("Total pages:", len(pages))


# Create chunks with page information
chunks = []

chunk_size = 1000
overlap = 200

for page_number, text in enumerate(pages, start=1):

    start = 0

    while start < len(text):

        end = start + chunk_size

        chunk_text = text[start:end].strip()

        if chunk_text:
            chunks.append({
                "page": page_number,
                "text": chunk_text
            })

        start += chunk_size - overlap


print("Total chunks:", len(chunks))
print("\nFirst chunk:")
print(chunks[0]["text"][:500])
print("\nPage:", chunks[0]["page"])

Total pages: 83
Total chunks: 299

First chunk:
1
Guiding Principles
for the Care of People with or at Risk for Diabetes

Page: 1


In [26]:
import nbformat
import os

NOTEBOOK3_PATH = (
    "/content/Medical-RAG-Hallucination-Detection/"
    "notebooks/Notebook_03_PDF_Text_Extraction.ipynb"
)

with open(NOTEBOOK3_PATH, "r", encoding="utf-8") as f:
    nb3 = nbformat.read(f, as_version=4)

print("Notebook 3 loaded")
print("Total cells:", len(nb3.cells))

Notebook 3 loaded
Total cells: 28


In [27]:
for i, cell in enumerate(nb3.cells):
    if cell.cell_type == "code" and (
        "chunk" in cell.source.lower()
        or "overlap" in cell.source.lower()
    ):
        print(f"\n========== CELL {i} ==========")
        print(cell.source)


========== CELL 15 ==========
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

print("Text splitter ready!")

========== CELL 16 ==========
chunks = []

for page in pages:
    page_chunks = splitter.split_text(page["clean_text"])

    for chunk_id, chunk in enumerate(page_chunks):
        chunks.append({
            "chunk_id": f"page_{page['page']}_chunk_{chunk_id}",
            "page": page["page"],
            "text": chunk
        })

print("Total chunks:", len(chunks))

========== CELL 17 ==========
print("Chunk ID:", chunks[0]["chunk_id"])
print("Page:", chunks[0]["page"])
print("Characters:", len(chunks[0]["text"]))
print("\nChunk:\n")
print(chunks[0]["text"])

========== CELL 18 ==========
print("Chunk ID:", chunks[100]["chunk_id"])
print("Page:", chunks[100]["page"])
print("Characters:", len(chunks[100]["text"]))
print("\nChunk:\n")
print(chunks[100]["text"])

========

In [29]:
for i in [14, 15, 16]:
    print(f"\n========== CELL {i} ==========")
    print(nb3.cells[i].source)


========== CELL 14 ==========
!pip install -q langchain-text-splitters

========== CELL 15 ==========
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

print("Text splitter ready!")

========== CELL 16 ==========
chunks = []

for page in pages:
    page_chunks = splitter.split_text(page["clean_text"])

    for chunk_id, chunk in enumerate(page_chunks):
        chunks.append({
            "chunk_id": f"page_{page['page']}_chunk_{chunk_id}",
            "page": page["page"],
            "text": chunk
        })

print("Total chunks:", len(chunks))


In [30]:
for i, cell in enumerate(nb3.cells):
    if cell.cell_type == "code" and (
        "clean_text" in cell.source
        or "pages =" in cell.source
    ):
        print(f"\n========== CELL {i} ==========")
        print(cell.source)


========== CELL 10 ==========
pages = []

for page_number, page in enumerate(doc, start=1):
    page_text = page.get_text().strip()

    pages.append({
        "page": page_number,
        "text": page_text
    })

print("Total pages processed:", len(pages))

========== CELL 12 ==========
import re

def clean_text(text):
    # Multiple spaces ko single space
    text = re.sub(r"[ \t]+", " ", text)

    # Multiple blank lines ko single newline
    text = re.sub(r"\n\s*\n+", "\n\n", text)

    # Har line ke beginning/end ke extra spaces
    text = "\n".join(line.strip() for line in text.splitlines())

    return text.strip()


for page in pages:
    page["clean_text"] = clean_text(page["text"])

print("Cleaning completed.")

========== CELL 13 ==========
print("Original characters:", len(pages[0]["text"]))
print("Cleaned characters:", len(pages[0]["clean_text"]))

print("\nCleaned page preview:\n")
print(pages[0]["clean_text"][:2000])

========== CELL 16 ==========
chunks = []

for page

In [32]:
!pip install -q pymupdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.8/25.8 MB 41.1 MB/s eta 0:00:00


In [35]:
import pymupdf

In [37]:
doc = pymupdf.open(PDF_PATH)

In [38]:
import pymupdf
import re
from langchain_text_splitters import RecursiveCharacterTextSplitter

doc = pymupdf.open(PDF_PATH)

pages = []

for page_number, page in enumerate(doc, start=1):
    page_text = page.get_text().strip()

    pages.append({
        "page": page_number,
        "text": page_text
    })

print("Total pages processed:", len(pages))


def clean_text(text):
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n\s*\n+", "\n\n", text)
    text = "\n".join(line.strip() for line in text.splitlines())
    return text.strip()


for page in pages:
    page["clean_text"] = clean_text(page["text"])


splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

chunks = []

for page in pages:
    page_chunks = splitter.split_text(page["clean_text"])

    for chunk_id, chunk in enumerate(page_chunks):
        chunks.append({
            "chunk_id": f"page_{page['page']}_chunk_{chunk_id}",
            "page": page["page"],
            "text": chunk
        })

print("Total chunks:", len(chunks))

Total pages processed: 83
Total chunks: 277


In [39]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

chunk_texts = [chunk["text"] for chunk in chunks]

embeddings = embedding_model.encode(
    chunk_texts,
    show_progress_bar=True
)

print("Embedding shape:", embeddings.shape)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/9 [00:00<?, ?it/s]

Embedding shape: (277, 384)


In [41]:
!pip install -q faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 22.2 MB/s eta 0:00:00


In [43]:
import faiss
import numpy as np

embeddings = np.array(embeddings).astype("float32")

index = faiss.IndexFlatL2(embeddings.shape[1])
index.add(embeddings)

print("FAISS index created")
print("Total vectors:", index.ntotal)

FAISS index created
Total vectors: 277


In [44]:
def retrieve_documents(question, top_k=5):
    """
    Retrieve the most relevant chunks from FAISS
    for the given question.
    """

    # Question ka embedding
    query_embedding = embedding_model.encode(
        question,
        normalize_embeddings=True
    )

    query_embedding = np.array(
        [query_embedding]
    ).astype("float32")

    # FAISS search
    distances, indices = index.search(
        query_embedding,
        top_k
    )

    retrieved_docs = []

    for rank, (idx, distance) in enumerate(
        zip(indices[0], distances[0]),
        start=1
    ):
        retrieved_docs.append({
            "rank": rank,
            "chunk_id": chunks[idx]["chunk_id"],
            "page": chunks[idx]["page"],
            "text": chunks[idx]["text"],
            "distance": float(distance)
        })

    return retrieved_docs


print("✅ Retrieval function created")

✅ Retrieval function created


In [45]:
question = "What are the risk factors for diabetes?"

retrieved_docs = retrieve_documents(
    question,
    top_k=5
)

print("Question:", question)
print("\nRetrieved Documents:\n")

for doc in retrieved_docs:
    print(
        f"Rank {doc['rank']} | "
        f"Page {doc['page']} | "
        f"Distance {doc['distance']:.4f}"
    )
    print(doc["text"][:300])
    print("-" * 80)

Question: What are the risk factors for diabetes?

Retrieved Documents:

Rank 1 | Page 5 | Distance 0.6568
5
INTRODUCTION
The diabetes problem
Today, 30.3 million people (9.4 percent of the U.S. population) have diabetes, including 7.2 million
who are undiagnosed.1 A major cause of blindness, renal failure, and amputation, diabetes
also increases the risk of cardiovascular disease, cancer, and dementia a
--------------------------------------------------------------------------------
Rank 2 | Page 9 | Distance 0.6770
9
Adapted from American Diabetes Association Standards of Care in Diabetes—2018
Risk of type 2 diabetes increases with age
and is strongly associated with overweight or
obesity—body mass index (BMI) ≥ 25 kg/m2
(≥ 23 kg/m2 for Asian Americans5)

Additional risk factors include
1.

2.

3.
4.
5.
Family
--------------------------------------------------------------------------------
Rank 3 | Page 5 | Distance 0.7260
by the Centers for Disease Control and Prevention (CDC) and o

In [55]:
from google import genai
from google.colab import userdata

API_KEY = userdata.get("Vivek28n")

client = genai.Client(api_key=API_KEY)

print("✅ Gemini client ready")

✅ Gemini client ready


In [56]:
def build_context(retrieved_docs):
    """
    Convert retrieved documents into a context
    for the Gemini model.
    """

    context_parts = []

    for doc in retrieved_docs:
        context_parts.append(
            f"[Source Page {doc['page']}]\n"
            f"{doc['text']}"
        )

    return "\n\n".join(context_parts)


context = build_context(retrieved_docs)

print(context[:3000])

[Source Page 5]
5
INTRODUCTION
The diabetes problem
Today, 30.3 million people (9.4 percent of the U.S. population) have diabetes, including 7.2 million
who are undiagnosed.1 A major cause of blindness, renal failure, and amputation, diabetes
also increases the risk of cardiovascular disease, cancer, and dementia and more than doubles
individual health care costs.2 The total estimated cost of diagnosed diabetes in 2017 was $327
billion, including $237 billion in direct medical costs and $90 billion in reduced productivity.2
Another 84.1 million Americans (33.9 percent of adults) have glucose levels that are higher than
normal but not high enough to be characterized as diabetes.1 Because persons with these glucose
levels are at increased risk of developing type 2 diabetes, this condition is termed prediabetes
by the Centers for Disease Control and Prevention (CDC) and other organizations.
Proper nutrition and physical activity are the cornerstones of treatment and prevention of type 2



In [57]:
def create_grounded_prompt(question, context):
    return f"""
You are a medical information assistant.

Answer the user's question using ONLY the provided evidence.

Rules:
1. Do not use outside knowledge.
2. Do not invent facts.
3. If the evidence is insufficient, say:
   "The provided document does not contain sufficient evidence to answer this question."
4. Mention the relevant source page numbers.
5. Do not provide personalized diagnosis or treatment.

Evidence:
{context}

Question:
{question}

Answer:
"""


prompt = create_grounded_prompt(
    question,
    context
)

print(prompt[:4000])


You are a medical information assistant.

Answer the user's question using ONLY the provided evidence.

Rules:
1. Do not use outside knowledge.
2. Do not invent facts.
3. If the evidence is insufficient, say:
   "The provided document does not contain sufficient evidence to answer this question."
4. Mention the relevant source page numbers.
5. Do not provide personalized diagnosis or treatment.

Evidence:
[Source Page 5]
5
INTRODUCTION
The diabetes problem
Today, 30.3 million people (9.4 percent of the U.S. population) have diabetes, including 7.2 million
who are undiagnosed.1 A major cause of blindness, renal failure, and amputation, diabetes
also increases the risk of cardiovascular disease, cancer, and dementia and more than doubles
individual health care costs.2 The total estimated cost of diagnosed diabetes in 2017 was $327
billion, including $237 billion in direct medical costs and $90 billion in reduced productivity.2
Another 84.1 million Americans (33.9 percent of adults) have

In [62]:
import time

for attempt in range(3):
    try:
        response = client.models.generate_content(
            model="gemini-3.6-flash",
            contents=prompt
        )

        answer = response.text

        print("Generated Answer:")
        print(answer)
        break

    except Exception as e:
        print(f"Attempt {attempt + 1} failed: {e}")

        if attempt < 2:
            print("Retrying in 5 seconds...")
            time.sleep(5)
        else:
            print("❌ Gemini request failed after 3 attempts.")

Attempt 1 failed: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}
Retrying in 5 seconds...
Generated Answer:
Based on the provided document, the risk factors for type 2 diabetes include:

* **Age:** The risk of type 2 diabetes increases with age [Source Page 9].
* **Overweight or Obesity:** Strongly associated with a body mass index (BMI) ≥ 25 kg/m² (or ≥ 23 kg/m² for Asian Americans) [Source Page 9].
* **Prediabetes:** Having glucose levels that are higher than normal, but not high enough to be characterized as diabetes [Source Page 5].
* **Family History:** Having a family history of diabetes (specifically a parent or sibling) [Source Page 9].
* **High-Risk Population:** Belonging to a high-risk population, including African American, Hispanic/Latino, American Indian, Alaska Native, Asian American, or Pacific Islander American [Source Page

In [63]:
rag_result = analyze_answer(
    answer,
    retrieved_docs
)

print("========== RAG ANSWER ANALYSIS ==========")

print("Semantic Similarity:",
      round(rag_result["semantic_similarity"], 4))

print("NLI Entailment:",
      round(rag_result["nli_entailment"], 4))

print("Evidence Support:",
      round(rag_result["support_score"], 4))

print("Detection Result:",
      rag_result["decision"])

========== RAG ANSWER ANALYSIS ==========
Semantic Similarity: 0.8639
NLI Entailment: 0.9956
Evidence Support: 0.9298
Detection Result: SUPPORTED


In [64]:
unsupported_answer = (
    "Type 2 diabetes can be prevented completely by "
    "taking vitamin D supplements every day."
)

unsupported_result = analyze_answer(
    unsupported_answer,
    retrieved_docs
)

print("========== UNSUPPORTED ANSWER ==========")

print("Semantic Similarity:",
      round(unsupported_result["semantic_similarity"], 4))

print("NLI Entailment:",
      round(unsupported_result["nli_entailment"], 4))

print("Evidence Support:",
      round(unsupported_result["support_score"], 4))

print("Detection Result:",
      unsupported_result["decision"])

========== UNSUPPORTED ANSWER ==========
Semantic Similarity: 0.5434
NLI Entailment: 0.0001
Evidence Support: 0.2718
Detection Result: POTENTIAL HALLUCINATION


In [69]:
import time

def medical_rag_with_hallucination_detection(
    question,
    top_k=5,
    max_retries=3
):
    """
    Complete RAG pipeline with hallucination detection.
    """

    # 1. Retrieve evidence
    retrieved_docs = retrieve_documents(
        question,
        top_k=top_k
    )

    # 2. Build context
    context = build_context(
        retrieved_docs
    )

    # 3. Create grounded prompt
    prompt = create_grounded_prompt(
        question,
        context
    )

    # 4. Generate answer with retry
    for attempt in range(max_retries):
        try:
            response = client.models.generate_content(
                model="gemini-3.6-flash",
                contents=prompt
            )

            answer = response.text
            break

        except Exception as e:
            print(
                f"Gemini attempt {attempt + 1} failed: {e}"
            )

            if attempt < max_retries - 1:
                print("Retrying in 5 seconds...")
                time.sleep(5)
            else:
                raise

    # 5. Detect hallucination
    detection = analyze_answer(
        answer,
        retrieved_docs
    )

    return {
        "question": question,
        "answer": answer,
        "retrieved_documents": retrieved_docs,
        "semantic_similarity": detection["semantic_similarity"],
        "nli_entailment": detection["nli_entailment"],
        "support_score": detection["support_score"],
        "decision": detection["decision"]
    }


print("✅ Complete RAG + Hallucination Detection pipeline created")

✅ Complete RAG + Hallucination Detection pipeline created


In [70]:
final_result = medical_rag_with_hallucination_detection(
    "What are the risk factors for diabetes?"
)

print("========== FINAL RESULT ==========")

print("\nQuestion:")
print(final_result["question"])

print("\nAnswer:")
print(final_result["answer"])

print("\nSemantic Similarity:",
      round(final_result["semantic_similarity"], 4))

print("NLI Entailment:",
      round(final_result["nli_entailment"], 4))

print("Evidence Support:",
      round(final_result["support_score"], 4))

print("Detection Result:",
      final_result["decision"])

print("\nSource Pages:")

pages = sorted(
    set(
        doc["page"]
        for doc in final_result["retrieved_documents"]
    )
)

print(pages)

Gemini attempt 1 failed: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}
Retrying in 5 seconds...
Gemini attempt 2 failed: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}
Retrying in 5 seconds...
========== FINAL RESULT ==========

Question:
What are the risk factors for diabetes?

Answer:
Based on the provided documents, the risk factors for type 2 diabetes include:

* **Age:** The risk of type 2 diabetes increases with age [Source Page 9].
* **Overweight or obesity:** Body mass index (BMI) ≥ 25 kg/m² (or ≥ 23 kg/m² for Asian Americans) [Source Page 9].
* **Prediabetes:** Having glucose levels that are higher than normal increases the risk of developing type 2 diabetes [Source Page 5].
* **Family